# 01 - Cargar los viajes: abrir los ZIP y armar la tabla bronze

**Antes de correr este notebook.** Databricks no se puede conectar a la página de la Intendencia. Por eso bajamos a mano el ZIP de cada mes desde el portal de datos abiertos y lo subimos al volume, en la carpeta `viajes_zip/`. El nombre del ZIP no importa: el notebook descubre el mes mirando el nombre del ZIP o el del CSV que tiene adentro.

**Parte A. Abrir los ZIP.** Saca el CSV de cada ZIP y lo guarda como `viajes/viajes_stm_AAAA_MM.csv`.

**Parte B. Tabla bronze.** Copia cada mes a la tabla `viajes_bronze` **sin limpiar nada**. Todas las columnas se guardan como texto y se agregan tres columnas para saber de dónde vino cada fila (mes, archivo y fecha de carga).

Se puede correr varias veces. Los meses que ya están hechos se saltean, salvo que se pida rehacerlos.

In [0]:
%run ./00_setup

# 00 - Preparar el proyecto

Este notebook deja todo listo para que los demás funcionen. Hace cuatro cosas:

1. Le dice a Python dónde está nuestro código (la carpeta `src/bondis`), para poder escribir `from bondis import ...`.
2. Crea en Databricks el lugar donde se guardan las tablas y los archivos (`bondis_stm.datos.raw`, definido en `config.py`).
3. Crea una carpeta para cada tipo de archivo (viajes, clima, etc.).
4. Muestra cómo dividimos los datos en train, val y test.

Se puede correr todas las veces que quieras. Si algo ya existe, no lo vuelve a crear ni lo borra.

No hace falta abrirlo a mano: los otros notebooks lo corren al principio con `%run ./00_setup`.

## 1. Encontrar nuestro código

Python solo puede importar código de las carpetas que conoce. Esta celda busca la carpeta del repositorio subiendo carpeta por carpeta hasta encontrar `src/bondis`, y la agrega a la lista de carpetas que Python conoce (`sys.path`).

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |
| `calendario/` | Por ahora vacía. El calendario está escrito en `src/bondis/calendario.py` |
| `paros/` | Días y horas con paro (todavía no lo hicimos) |

## 4. Revisar que todo esté bien

Muestra la versión de Spark, el usuario y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

## Opciones

Antes de correr, elegí qué hacer cambiando estos valores en la celda de abajo:

| Opción | Valor | Qué pasa |
|---|---|---|
| `MESES` | `["2024-01"]` | Procesa solo esos meses. **La primera vez, probá con un solo mes.** |
| `MESES` | `None` | Procesa todos los meses que estén subidos. |
| `FUENTE` | `"zip"` | Abre los ZIP de `viajes_zip/` (Parte A) y después carga bronze. |
| `FUENTE` | `"volume"` | Se saltea la Parte A, porque los CSV ya están en `viajes/`. |
| `FORZAR_BRONZE` | `True` | Vuelve a cargar meses que ya estaban en bronze. |

In [0]:
MESES = None             # None quiere decir todos los meses
FUENTE = "zip"           # "zip" o "volume"
CARPETA_ZIP = config.RAW_ZIP   # cambiar si subiste los ZIP a otra carpeta
FORZAR_BRONZE = False
 
assert FUENTE in ("zip", "volume"), "FUENTE debe ser 'zip' o 'volume'"
CARPETA_CSV = config.RAW_VIAJES

In [0]:
import importlib
from pathlib import Path
 
from pyspark.sql import functions as F
 
# Funciona porque antes corrimos %run ./00_setup, que le dice a Python dónde está src/
from bondis import ingesta
 
importlib.reload(ingesta)
 
 
def mes_elegido(anio_mes):
    """Verdadero si hay que procesar el mes 'AAAA-MM' según la opción MESES."""
    return MESES is None or anio_mes in MESES

## Parte A. Abrir los ZIP que bajamos a mano

Para cada ZIP de la carpeta:
1. Descubre de qué mes es, por el nombre del ZIP (o, si no se puede, por el nombre del CSV de adentro).
2. Saca el CSV y lo guarda como `raw/viajes/viajes_stm_AAAA_MM.csv`.
3. Revisa que el CSV que sacó tenga el mismo tamaño que dice el ZIP. Si no, el archivo quedó cortado.

Los ZIP no se borran. Si un CSV ya estaba sacado, se saltea.

**Mirá la tabla de meses de la próxima celda antes de seguir.** Ahí se ve si hay meses repetidos o meses que faltan.

In [0]:
if FUENTE == "zip":
    # Busca también en subcarpetas, y acepta .zip, .ZIP o .Zip
    zips = sorted(p for p in Path(CARPETA_ZIP).rglob("*") if p.is_file() and p.suffix.lower() == ".zip")
    print(f"ZIP encontrados en {CARPETA_ZIP}: {len(zips)}\n")
 
    # Primero solo mostramos qué mes le toca a cada ZIP, sin sacar nada todavía.
    # Si el nombre del ZIP no tiene el mes, se mira el nombre del CSV de adentro.
    mes_por_zip = {}
    print(f"{'ZIP':<34} {'Mes':<8} {'ZIP':>8} {'CSV':>8}  CSV interno")
    for z in sorted(zips, key=lambda p: ingesta.mes_de_zip(p) or "9999"):
        am = ingesta.mes_de_zip(z)
        mes_por_zip[z] = am
        internos = ingesta.contenido_zip(z)
        nombre_csv = internos[0][0] if internos else "(sin CSV)"
        mb_csv = sum(mb for _, mb in internos)
        print(f"{z.name:<34} {am or '¿?':<8} {z.stat().st_size / 1e6:>6,.0f}MB {mb_csv:>6,.0f}MB  {nombre_csv}")
 
    # Revisamos si hay meses repetidos o meses que faltan entre 2024 y 2026.
    # Los meses de 2026 que todavía no publicó la Intendencia van a salir como faltantes, es normal.
    from collections import Counter
    conteo = Counter(am for am in mes_por_zip.values() if am)
    repetidos = sorted(am for am, n in conteo.items() if n > 1)
    esperados = (
        spark.sql(f"SELECT explode(sequence(to_date('{config.FECHA_INICIO}'), "
                  f"date_sub(to_date('{config.FECHA_FIN}'), 1), interval 1 month)) AS d")
        .selectExpr("date_format(d, 'yyyy-MM') AS am").toPandas()["am"].tolist()
    )
    faltantes = [am for am in esperados if am not in conteo]
    print(f"\nMeses identificados: {len(conteo)}  |  sin identificar: {sum(1 for a in mes_por_zip.values() if not a)}")
    print(f"Meses repetidos: {repetidos or 'ninguno'}")
    print(f"Meses del período sin ZIP: {faltantes or 'ninguno'}")

ZIP encontrados en /Volumes/bondis_stm/datos/raw/viajes_zip: 0

ZIP                                Mes           ZIP      CSV  CSV interno

Meses identificados: 0  |  sin identificar: 0
Meses repetidos: ninguno
Meses del período sin ZIP: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10', '2026-11', '2026-12']


In [0]:
errores_zip = []
 
if FUENTE == "zip":
    for z in zips:
        am = ingesta.mes_desde_texto(z.stem)
        if am is not None and not mes_elegido(am):
            continue
        try:
            am, _ = ingesta.descomprimir_zip(z, config.RAW_VIAJES)
        except Exception as e:  # si un ZIP falla, se anota el error y se sigue con el próximo
            errores_zip.append((z.name, str(e)))
            print(f"[FAIL] {z.name}: {e}")
 
    print(f"\nErrores: {len(errores_zip)}")
    for nombre, err in errores_zip:
        print(f"  {nombre}: {err}")


Errores: 0


## Parte B. Cargar a bronze
### B.1 Qué archivos hay y qué meses ya están cargados

In [0]:
archivos = sorted(Path(CARPETA_CSV).glob("viajes_stm_*.csv"))
print(f"Carpeta de origen: {CARPETA_CSV}  ({len(archivos)} CSV de viajes en total)")
por_mes = {am: ps for am, ps in ingesta.agrupar_por_mes(archivos).items() if mes_elegido(am)}
 
tabla_existe = spark.catalog.tableExists(config.TABLE_BRONZE)
cargados = set()
if tabla_existe:
    cargados = {
        r["anio_mes"]
        for r in spark.table(config.TABLE_BRONZE).select("anio_mes").distinct().collect()
    }
 
pendientes = {am: ps for am, ps in por_mes.items() if FORZAR_BRONZE or am not in cargados}
 
print(f"Meses con CSV en el volume: {len(por_mes)}")
print(f"Meses ya cargados en bronze: {len(cargados & set(por_mes))}")
print(f"Meses a cargar ahora:        {len(pendientes)}  {list(pendientes)}")

Carpeta de origen: /Volumes/bondis_stm/datos/raw/viajes  (0 CSV de viajes en total)
Meses con CSV en el volume: 0
Meses ya cargados en bronze: 0
Meses a cargar ahora:        0  []


### B.2 Cargar mes por mes

Para cada mes:
1. Mira el principio del archivo para saber qué símbolo separa las columnas (coma, punto y coma...) y cómo están guardadas las tildes.
2. Lee el CSV con **todas las columnas como texto**. En bronze no decidimos si algo es número o fecha, eso se hace en silver.
3. Deja prolijos los nombres de las columnas (sin tildes ni espacios).
4. Agrega `anio_mes`, `archivo_origen` y `fecha_ingesta`, para saber de dónde salió cada fila.
5. Guarda el mes en la tabla. Si el mes ya estaba, **reemplaza solo ese mes** y deja los demás como estaban (eso hace `replaceWhere`).

In [0]:
columnas_por_mes = {}
 
for am, paths in pendientes.items():
    sep, enc, encabezado = ingesta.detectar_formato(paths[0])
    print(f"[{am}] {len(paths)} archivo/s, separador {sep!r}, {enc}")
 
    crudo = (
        spark.read
        .option("header", True)
        .option("sep", sep)
        .option("encoding", enc)
        .option("quote", '"')
        .option("escape", '"')
        .option("inferSchema", False)
        .csv(paths)
    )
 
    nuevos_nombres = ingesta.normalizar_columnas(crudo.columns)
    df = crudo.select(
        *[F.col(f"`{c}`").alias(n) for c, n in zip(crudo.columns, nuevos_nombres)],
        F.lit(am).alias("anio_mes"),
        F.col("_metadata.file_name").alias("archivo_origen"),
        F.current_timestamp().alias("fecha_ingesta"),
    )
 
    escritor = df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if tabla_existe:
        escritor.option("replaceWhere", f"anio_mes = '{am}'").saveAsTable(config.TABLE_BRONZE)
    else:
        escritor.partitionBy("anio_mes").saveAsTable(config.TABLE_BRONZE)
        tabla_existe = True
 
    filas = spark.table(config.TABLE_BRONZE).where(F.col("anio_mes") == am).count()
    columnas_por_mes[am] = nuevos_nombres
    print(f"        {filas:,} filas, columnas: {nuevos_nombres}")

### B.3 ¿Todos los meses tienen las mismas columnas?

Si la Intendencia cambió el formato de un año a otro, algunos meses pueden traer columnas distintas. Bronze las guarda todas igual (eso hace `mergeSchema`). Si pasa, se arregla en silver.

In [0]:
if columnas_por_mes:
    esquemas = {}
    for am, cols in columnas_por_mes.items():
        esquemas.setdefault(tuple(cols), []).append(am)
    if len(esquemas) == 1:
        print("Todos los meses cargados tienen las mismas columnas.")
    else:
        print(f"ATENCIÓN: {len(esquemas)} esquemas distintos.")
        for cols, meses in esquemas.items():
            print(f"  Meses {meses}:\n    {list(cols)}")
else:
    print("No se cargaron meses en esta ejecución.")

No se cargaron meses en esta ejecución.


## Revisión final

Cuántas filas y archivos quedaron por mes, y cómo se ven las primeras filas.

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    display(
        spark.table(config.TABLE_BRONZE)
        .groupBy("anio_mes")
        .agg(F.count("*").alias("filas"), F.countDistinct("archivo_origen").alias("archivos"))
        .orderBy("anio_mes")
    )

anio_mes,filas,archivos
2024-01,19054962,1
2024-02,19966497,1
2024-03,23408053,1
2024-04,26296998,1
2024-05,26281987,1
2024-06,25182931,1
2024-07,24547115,1
2024-08,26079619,1
2024-09,25615090,1
2024-10,28214243,1


In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    spark.table(config.TABLE_BRONZE).printSchema()
    display(spark.table(config.TABLE_BRONZE).limit(10))

root
 |-- id_viaje: string (nullable = true)
 |-- con_tarjeta: string (nullable = true)
 |-- fecha_evento: string (nullable = true)
 |-- tipo_viaje: string (nullable = true)
 |-- descripcion_tipo_viaje: string (nullable = true)
 |-- grupo_usuario: string (nullable = true)
 |-- descripcion_grupo_usuario: string (nullable = true)
 |-- grupo_usuario_especifico: string (nullable = true)
 |-- descripcion_grupo_usuario_espe: string (nullable = true)
 |-- ordinal_de_tramo: string (nullable = true)
 |-- cantidad_pasajeros: string (nullable = true)
 |-- codigo_parada_origen: string (nullable = true)
 |-- cod_empresa: string (nullable = true)
 |-- descrip_empresa: string (nullable = true)
 |-- linea_codigo: string (nullable = true)
 |-- dsc_linea: string (nullable = true)
 |-- sevar_codigo: string (nullable = true)
 |-- anio_mes: string (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- fecha_ingesta: timestamp (nullable = true)



id_viaje,con_tarjeta,fecha_evento,tipo_viaje,descripcion_tipo_viaje,grupo_usuario,descripcion_grupo_usuario,grupo_usuario_especifico,descripcion_grupo_usuario_espe,ordinal_de_tramo,cantidad_pasajeros,codigo_parada_origen,cod_empresa,descrip_empresa,linea_codigo,dsc_linea,sevar_codigo,anio_mes,archivo_origen,fecha_ingesta
5922543,1,2024-01-15 07:57:19,11,JUB. B,3,JUBILADO,32,JUBILADO B,2,1,4007,50,C.U.T.C.S.A.,40,121,7884,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
1512174,1,2024-01-04 09:38:34,11,JUB. B,3,JUBILADO,32,JUBILADO B,2,1,2225,50,C.U.T.C.S.A.,78,169,4710,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
5186499,1,2024-01-12 16:06:27,11,JUB. B,3,JUBILADO,32,JUBILADO B,2,1,1139,50,C.U.T.C.S.A.,76,163,4310,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
9838588,1,2024-01-22 16:18:26,11,JUB. B,3,JUBILADO,32,JUBILADO B,2,1,1139,50,C.U.T.C.S.A.,76,163,4310,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
342996,1,2024-01-02 10:27:48,8,EST. B,2,ESTUDIANTE,22,ESTUDIANTE B,2,1,563,50,C.U.T.C.S.A.,496,130,4753,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
8090739,1,2024-01-18 09:39:34,8,EST. B,2,ESTUDIANTE,22,ESTUDIANTE B,2,1,3461,50,C.U.T.C.S.A.,85,181,7603,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
2808289,1,2024-01-08 17:01:35,11,JUB. B,3,JUBILADO,32,JUBILADO B,2,1,3530,10,C.O.E.T.C.,3,405,8782,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
4265536,1,2024-01-11 10:06:26,10,JUB. A,3,JUBILADO,31,JUBILADO A,2,1,3735,20,C.O.M.E.S.A,16,522,2426,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
10008999,1,2024-01-23 10:53:42,11,JUB. B,3,JUBILADO,32,JUBILADO B,2,1,1567,70,U.C.O.T.,134,370,4400,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
13208713,1,2024-01-29 18:26:15,10,JUB. A,3,JUBILADO,31,JUBILADO A,2,1,3358,50,C.U.T.C.S.A.,76,163,4310,2024-01,viajes_stm_2024_01.csv,2026-10-30T17:07:32.419Z
